In [ ]:
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
import json
import re
from collections import defaultdict

In [ ]:
base_price_ranges = {
    "low": (1, 10),
    "medium": (10, 100),
    "high": (100, 1000),
    "very_high": (1000, 10000)
}

In [ ]:
def scenario_generation_system_fn(**kwargs):
    return """You are an assistant for a bargaining research project.
Your task is write scenarios for bargaining problems where two agents, a seller and a buyer, try to make a deal over the price of a product."""

def scenario_generation_user_prompt_fn(base_price_range, currency="$", num_scenarios=1):
    prompt_str = f"""Please think of a product that a seller and a buyer may want to trade. Think of a price range of approx. {currency}{base_price_range[0]} - {currency}{base_price_range[1]}. The product must be a commodity which can be also bought and sold on the open market.
First write an extended scenario description in English, then put it in a JSON format, using the following fields:
"product_name", "product_description" (2-3 sentences), "buyer_persona" (2-3 sentences, written in 2nd person), "seller_persona" (2-3 sentences, written in 2nd person), "price_floor", "price_ceiling"
"""
    if num_scenarios > 1:
        prompt_str += f"\nGenerate {num_scenarios} diverse scenarios, separated by a # on a single line"
    return prompt_str

scenario_generation_in_context_example_0_user = \
"""Please think of a product that a seller and a buyer may want to trade. Think of a price range of approx. $1 - $10. The product must be a commodity which can be also bought and sold on the open market.
First write an extended scenario description in English, then put it in JSON format, using the following fields:
"product_name", "product_description" (2-3 sentences), "buyer_persona" (2-3 sentences, written in 2nd person), "seller_persona" (2-3 sentences, written in 2nd person), "price_floor", "price_ceiling"
"""

scenario_generation_in_context_example_0_assistant = \
"""Product: One dozen fresh eggs

Market context: Eggs are a common commodity, widely bought and sold on the open market such as supermarkets, farmers’ markets, and wholesalers.

Typical price range: $2.50 – $4.50 per dozen, depending on factors like freshness, farming practices (free range, organic), and location.

Scenario:

- The seller is a local farmer who has extra eggs and wants to sell a dozen.
- The buyer is a resident seeking fresh, local eggs.
- Both know that nearby stores sell non-local eggs for around $2.50 per dozen, while local organic options can cost up to $4.50.
- They begin negotiating with these reference points in mind.

```json
{
  "product_name": "One dozen fresh eggs",
  "product_description": "A dozen fresh eggs sourced from local chickens. Eggs are a widely traded commodity, available at supermarkets and farmers’ markets, with prices varying based on quality and sourcing.",
  "buyer_persona": "You are a resident looking for fresh, local eggs and prefer to support nearby farmers. You are aware of supermarket prices but value higher freshness and local sourcing. You are willing to pay a bit extra compared to store prices if the quality is good.",
  "seller_persona": "You are a local farmer with a surplus of fresh eggs and are looking to sell directly to consumers. You want a fair price for your high-quality, locally produced eggs, possibly higher than wholesale or supermarket rates. You value building relationships with local buyers while earning a reasonable profit.",
  "price_floor": 2.50,
  "price_ceiling": 4.50
}
```
"""

In [ ]:
def scenario_generation_to_openai_api_format(scenario_params, model_params):
    messages = [{"role": "system", "content": scenario_generation_system_fn(**scenario_params)}, 
                {"role": "user", "content": scenario_generation_in_context_example_0_user},
                {"role": "assistant", "content": scenario_generation_in_context_example_0_assistant},
                {"role": "user", "content": scenario_generation_user_prompt_fn(**scenario_params)}]
    rv = model_params.copy()
    rv["messages"] = messages
    return json.dumps(rv)

In [ ]:
model_name = "gpt-4.1-2025-04-14"
model_provider = "openai"

model_params = {
        "model": model_name,
        "temperature": 1.0,
        "top_p": 1.0,
        "max_tokens": 32768,
        "n": 16
    }

num_scenarios = 16
num_repeat_queries_per_base_price_range = 4

base_filename = "./scenario_generation"


In [ ]:
filename_to_llm = base_filename+"_to_"+model_provider+".jsonl"
filename_from_llm = base_filename+"_from_"+model_provider+".jsonl"

In [ ]:
with open(filename_to_llm, "w") as out_fs:
    for k, base_price_range in base_price_ranges.items():
        scenario_params = {
            "base_price_range": base_price_range,
            "currency": "$",
            "num_scenarios": num_scenarios
        }
        for j in range(num_repeat_queries_per_base_price_range):
            print(scenario_generation_to_openai_api_format(scenario_params, model_params), file=out_fs)

In [ ]:
with open(filename_from_llm, "r") as in_fs:
    from_llm_entries = [json.loads(line) for line in in_fs.readlines()]

inv_base_price_ranges = {v: k for k, v in base_price_ranges.items()}
raw_scenarios_by_base_price_range = defaultdict(list)
for entry in from_llm_entries:
    user_message = entry[0]["messages"][3]["content"]
    match = re.search(r'\$([\d.]+)\s*-\s*\$([\d.]+)', user_message)
    try:
        base_price_floor = float(match.group(1))
        base_price_ceiling = float(match.group(2))
        base_price_class = inv_base_price_ranges[(base_price_floor, base_price_ceiling)]
        for choice in entry[1]["choices"]:
            assistant_message = choice["message"]["content"]
            for raw_scenario_str in assistant_message.split("#\n"):
                raw_scenario_str = raw_scenario_str.strip()
                if len(raw_scenario_str) > 0:
                    raw_scenarios_by_base_price_range[base_price_class].append(raw_scenario_str)
    except:
        continue

scenarios_by_base_price_range = defaultdict(list)
for base_price_class, raw_scenarios in raw_scenarios_by_base_price_range.items():
    base_price_floor, base_price_ceiling = base_price_ranges[base_price_class]
    print(f"Base price class: '{base_price_class}' (${base_price_floor} - ${base_price_ceiling}). Num. raw scenarios: {len(raw_scenarios)}")
    for i, raw_scenario in enumerate(raw_scenarios):
        match = re.search(r"```json\s*(\{.*?\})\s*```", raw_scenario, re.DOTALL)
        try:
            json_str = match.group(1)
            scenario_dict = json.loads(json_str)
            if not (("product_name" in scenario_dict) and ("product_description" in scenario_dict) and ("buyer_persona" in scenario_dict) and ("seller_persona" in scenario_dict) and ("price_floor" in scenario_dict) and ("price_ceiling" in scenario_dict)):
                print(f"Missing field in scenario {base_price_class}, {i}:\n{scenario_dict}")
                continue
            price_floor = scenario_dict["price_floor"]
            price_ceiling = scenario_dict["price_ceiling"]
            if (price_ceiling <= price_floor):
                print(f"Bad price range in scenario {base_price_class}, {i}:\n{scenario_dict}")
                continue
            # compute calculated fields
            scenario_dict["midpoint_price"] = (price_ceiling + price_floor) * 0.5
            scenario_dict["price_spread"] = price_ceiling - price_floor
            scenarios_by_base_price_range[base_price_class].append(scenario_dict)
        except Exception as e:
            #print(f"Parsing error in scenario {base_price_class}, {i}: {type(e)}\n{str(e)}")
            #print(raw_scenario)
            #break
            continue

for base_price_class, scenarios in scenarios_by_base_price_range.items():
    base_price_floor, base_price_ceiling = base_price_ranges[base_price_class]
    print(f"Base price class: '{base_price_class}' (${base_price_floor} - ${base_price_ceiling}). Num. scenarios: {len(scenarios)}")

In [ ]:
scenarios_filename = "./scenarios.jsonl"

In [ ]:
with open(scenarios_filename, "w") as out_fs:
    json.dump(scenarios_by_base_price_range, out_fs, indent=2)

In [ ]:
# Split scenarios price ranges

In [ ]:
# Reload

with open(scenarios_filename) as in_fs:
    scenarios_by_base_price_range = json.load(in_fs)

In [ ]:
scenarios_by_base_price_range["low"][0]

In [ ]:
scenarios_by_reservation_ranges = dict()

for k, v in scenarios_by_base_price_range.items():
    new_v = []
    for entry in v:
        new_entry = dict(entry)
        del new_entry["price_floor"]
        del new_entry["price_ceiling"]
        del new_entry["midpoint_price"]
        del new_entry["price_spread"]
        new_entry["seller_res_price_range"] = (entry["price_floor"], entry["midpoint_price"])
        new_entry["buyer_res_price_range"] = (entry["midpoint_price"], entry["price_ceiling"])
        new_v.append(new_entry)
    scenarios_by_reservation_ranges[k] = new_v

In [ ]:
scenarios_by_reservation_ranges_filename = "./scenarios_by_reservation_ranges.jsonl"

In [ ]:
with open(scenarios_by_reservation_ranges_filename, "w") as out_fs:
    json.dump(scenarios_by_reservation_ranges, out_fs, indent=2)